In [2]:
import sys
import os
import requests
from datetime import datetime, timedelta
from fastapi import FastAPI, Query
from fastapi.testclient import TestClient

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from shared.config import Config

 Using model: deepseek/deepseek-v3.2


In [3]:
def get_weather(location: str, period: str = "current", dt: str = None):
    api_key = Config.Weather.OPENWEATHER_API_KEY
    base_url = "http://api.weatherapi.com/v1"
    
    if period == "forecast":
        url = f"{base_url}/forecast.json?key={api_key}&q={location}&days=7&lang=th"
    elif period == "historical":
        if not dt:
            dt = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
        url = f"{base_url}/history.json?key={api_key}&q={location}&dt={dt}&lang=th"
    else:
        url = f"{base_url}/current.json?key={api_key}&q={location}&lang=th"

    try:
        response = requests.get(url)
        data = response.json()
        
        if "error" in data:
            return {"error": data["error"]["message"]}
            
        return {
            "display_name": f"{data.get('location', {}).get('name', 'Unknown')}, {data.get('location', {}).get('country', 'Unknown')}",
            
            "temp": data.get("current", {}).get("temp_c") if period == "current" else f"See {period} data",
            "condition": data.get("current", {}).get("condition", {}).get("text") if period == "current" else f"See {period} data",
            
            "full_data": data
        }
    except Exception as e:
        return {"error": str(e)}

In [4]:
app = FastAPI()

@app.get("/weather")
def weather_endpoint(
    location: str, 
    period: str = Query("current", enum=["current", "forecast", "historical"]),
    dt: str = Query(None, description="Date in YYYY-MM-DD format for historical data")
):
    return get_weather(location, period, dt)


client = TestClient(app)

In [5]:
print("Current Weather")
res_current = client.get("/weather", params={"location": "Bangkok"})
print(res_current.json(), "\n")

print("2. Forecast")
res_forecast = client.get("/weather", params={"location": "Phuket", "period": "forecast"})
print(res_forecast.json(), "\n")

print(" 3. Historical")
res_history = client.get("/weather", params={"location": "Japan", "period": "historical", "dt": "2024-03-20"})
print(res_history.json())


Current Weather
{'display_name': 'Bangkok, Thailand', 'temp': 30.1, 'condition': 'Clear', 'full_data': {'location': {'name': 'Bangkok', 'region': 'Krung Thep', 'country': 'Thailand', 'lat': 13.75, 'lon': 100.5167, 'tz_id': 'Asia/Bangkok', 'localtime_epoch': 1775833429, 'localtime': '2026-04-10 22:03'}, 'current': {'last_updated_epoch': 1775833200, 'last_updated': '2026-04-10 22:00', 'temp_c': 30.1, 'temp_f': 86.2, 'is_day': 0, 'condition': {'text': 'Clear', 'icon': '//cdn.weatherapi.com/weather/64x64/night/113.png', 'code': 1000}, 'wind_mph': 14.8, 'wind_kph': 23.8, 'wind_degree': 189, 'wind_dir': 'S', 'pressure_mb': 1009.0, 'pressure_in': 29.8, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 75, 'cloud': 25, 'feelslike_c': 33.6, 'feelslike_f': 92.5, 'windchill_c': 30.0, 'windchill_f': 86.1, 'heatindex_c': 33.5, 'heatindex_f': 92.3, 'dewpoint_c': 21.9, 'dewpoint_f': 71.5, 'vis_km': 10.0, 'vis_miles': 6.0, 'uv': 0.0, 'gust_mph': 18.9, 'gust_kph': 30.3, 'short_rad': 0, 'diff_rad': 0, 'dn

In [6]:
get_weather(location="Japan", period="current")

{'display_name': 'Tokyo, Japan',
 'temp': 20.4,
 'condition': 'Partly Cloudy',
 'full_data': {'location': {'name': 'Tokyo',
   'region': 'Tokyo',
   'country': 'Japan',
   'lat': 35.69,
   'lon': 139.692,
   'tz_id': 'Asia/Tokyo',
   'localtime_epoch': 1775833430,
   'localtime': '2026-04-11 00:03'},
  'current': {'last_updated_epoch': 1775833200,
   'last_updated': '2026-04-11 00:00',
   'temp_c': 20.4,
   'temp_f': 68.7,
   'is_day': 0,
   'condition': {'text': 'Partly Cloudy',
    'icon': '//cdn.weatherapi.com/weather/64x64/night/116.png',
    'code': 1003},
   'wind_mph': 3.8,
   'wind_kph': 6.1,
   'wind_degree': 246,
   'wind_dir': 'WSW',
   'pressure_mb': 998.0,
   'pressure_in': 29.47,
   'precip_mm': 0.0,
   'precip_in': 0.0,
   'humidity': 88,
   'cloud': 0,
   'feelslike_c': 20.4,
   'feelslike_f': 68.7,
   'windchill_c': 17.8,
   'windchill_f': 64.0,
   'heatindex_c': 17.8,
   'heatindex_f': 64.0,
   'dewpoint_c': 14.9,
   'dewpoint_f': 58.7,
   'vis_km': 10.0,
   'vis_mile